# 03. Chunking Experiments

This notebook explores and compares different chunking strategies for the MS MARCO dataset. Since passages in the dataset can be large, effective chunking is critical to building a high-performing retrieval system. We will evaluate original, fixed-size, sentence-aware, and metadata-aware chunking.

In [ ]:
import sys
import os
import json
import time
import matplotlib.pyplot as plt

# Add repository root to path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from colab.src import utils
from colab.src import dataset_utils
from colab.src import chunking

# Setup environment
utils.set_seed(42)
config = utils.load_config('experiment_config.yaml')
utils.print_header("Chunking Experiments Setup Complete")

## 1. Load Dataset
Extracting all passages for chunking experiments.

In [ ]:
print("Loading MS MARCO dataset...")
dataset = dataset_utils.load_msmarco_xi(config)
passages = dataset_utils.extract_all_passages(dataset)
print(f"Extracted {len(passages)} passages.")

## 2. Strategy 1: Original Passage Chunking
This baseline strategy uses the passages as they are, treating each passage as a single chunk.

In [ ]:
print("Running Original Passage Chunking...")
start_time = time.time()
result_original = chunking.chunk_original(passages)
time_original = time.time() - start_time
print(f"Completed in {time_original:.2f} seconds.")
print(f"Chunks: {result_original.num_chunks}, Avg Size: {result_original.avg_chunk_size:.2f}")

## 3. Strategy 2: Fixed-size Chunking
This strategy cuts the text into equal-sized parts, with configurable overlap. We use a chunk size of 700 characters and an overlap of 100.

In [ ]:
print("Running Fixed-size Chunking (700/100)...")
start_time = time.time()
result_fixed = chunking.chunk_fixed(passages, chunk_size=700, overlap=100)
time_fixed = time.time() - start_time
print(f"Completed in {time_fixed:.2f} seconds.")
print(f"Chunks: {result_fixed.num_chunks}, Avg Size: {result_fixed.avg_chunk_size:.2f}")

## 4. Strategy 3: Sentence-aware Chunking
This strategy splits text intelligently at sentence boundaries, respecting natural language breaks rather than cutting mid-word.

In [ ]:
print("Running Sentence-aware Chunking...")
start_time = time.time()
result_sentence = chunking.chunk_sentence(passages)
time_sentence = time.time() - start_time
print(f"Completed in {time_sentence:.2f} seconds.")
print(f"Chunks: {result_sentence.num_chunks}, Avg Size: {result_sentence.avg_chunk_size:.2f}")

## 5. Strategy 4: Metadata-aware Chunking
This strategy incorporates document metadata (like IDs or structural cues) directly into the chunk text so the retrieval model has more context.

In [ ]:
print("Running Metadata-aware Chunking...")
start_time = time.time()
result_metadata = chunking.chunk_metadata_aware(passages)
time_metadata = time.time() - start_time
print(f"Completed in {time_metadata:.2f} seconds.")
print(f"Chunks: {result_metadata.num_chunks}, Avg Size: {result_metadata.avg_chunk_size:.2f}")

## 6. Comparison and Visualization
We'll compare the results of all strategies to determine the most effective approach.

In [ ]:
results = {
    "Original": result_original,
    "Fixed-size": result_fixed,
    "Sentence-aware": result_sentence,
    "Metadata-aware": result_metadata
}

comparison = chunking.compare_strategies(results)
utils.print_table(comparison, title="Chunking Strategies Comparison")

# Visualization
strategies = list(results.keys())
avg_sizes = [res.avg_chunk_size for res in results.values()]

plt.figure(figsize=(10, 6))
plt.bar(strategies, avg_sizes, color=['blue', 'orange', 'green', 'red'])
plt.title('Average Chunk Size Distribution Across Strategies')
plt.ylabel('Average Size (characters)')
plt.xlabel('Strategy')
plt.show()

## Note on Semantic Chunking
> **Why isn't semantic chunking here?** Semantic chunking requires an embedding model to measure meaning similarity between sentences and group them accordingly. This strategy is deferred to **Notebook 07**, which follows the embedding model selection pipeline.

## 7. Save Results

In [ ]:
reports_dir = utils.get_reports_dir()
report_path = os.path.join(reports_dir, 'chunking_comparison.json')

# Prepare serializable dict
serializable_results = {k: v.__dict__ if hasattr(v, '__dict__') else v for k, v in comparison.items()}
utils.save_json(serializable_results, report_path)
print(f"Results saved to {report_path}")

## Decision
Based on these findings, we can decide on a primary strategy for indexing. 

- **Original** works best for small average passage sizes.
- **Sentence-aware** avoids destructive cuts and is generally the safest baseline for diverse text lengths.
- **Fixed-size** provides uniform text boundaries and works reliably with maximum sequence lengths of certain transformer models.

*We will carry these chunk configurations forward into the retrieval benchmarks to determine which empirically yields higher NDCG@10 and Recall.*